<a href="https://colab.research.google.com/github/maheen24-cell/ees322_lab1_maheen/blob/main/Lab_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#select the reflectance bands once for clarity
green = image.select("SR_B3")
red = image.select("SR_B4")
nir = image.select("SR_B5")
swir2 = image.select("SR_B7")

#NDVI
ndvi = image.expression(
    "(NIR - RED)/(NIR + RED)",
    {"NIR":nir, "RED":red}
).rename("NDVI")

#NDWI
ndwi = image.expression(
  "(GREEN - NIR)/(GREEN + NIR)",
  {"GREEN": green, "NIR": nir}
).rename("NDWI")

#SAVI
L = 0.5
savi = image.expression(
    "((NIR - RED)/(NIR + RED + L))*(1 + L)",
    {"NIR":nir, "RED":red, "L":L}
).rename("SAVI")

#NBR
nbr = image.expression(
    "(NIR - SWIR2)/(NIR + SWIR2)",
    {"NIR":nir, "SWIR2":swir2}
).rename("NBR")

index_image = image.addBands([ndvi, ndwi, savi, nbr])
print("Index bands added:")
print(index_image.bandNames().getInfo())

In [ ]:
# Visualizing the Maps
Map = geemap.Map()
Map.centerObject(aoi, 10)

 true_color = {
     "bands":['SR_B4', 'SR_B3', 'SR_B2'],
     "min": 0.0,
     "max": 0.60
 }

 false_color = {
     "bands":['SR_B5', 'SR_B4', 'SR_B3'],
     "min": 0.0,
     "max": 0.30
 }

 Map.addLayer(image.clip(aoi), true_color, "True Color Composite", True)
 Map.addLayer(image.clip(aoi), false_color, "False Color Composite", False)

 Map.addLayer(
     ndvi.clip(aoi),
     {"min":0.25, "max":0.72, "palette":['#faffec', '#e2ff8e', '#a8c557']},
     "NDVI",
     False,
 )

 Map.addLayer(
     ndwi.clip(aoi),
     {"min":-1, "max":1, "palette":['a6611a', 'f5f5f5', '018571']},
      "NDWI",
     False,
 )

 Map.addLayer(
     savi.clip(aoi),
     {"min":-1, "max":1, "palette":['8c510a', 'f6e8c3', '01655e']},
     "SAVI",
     False,
 )

Map.addLayer(
    nbr.clip(aoi),
    {"min":-1, "max":1, "palette":['b2182b', 'f7f7f7', '2166ac']},
    "NBR",
    False,
)

Map.addLayer(aoi, {"color":"black"}, "AOI", False)
Map

In [ ]:
# Part G — Quantitative evaluation

indices = index_image.select(["NDVI", "NDWI", "SAVI", "NBR"])

stats = indices.reduceRegion(
    reducer=(
        ee.Reducer.mean()
        .combine(ee.Reducer.stdDev(), sharedInputs=True)
        .combine(ee.Reducer.minMax(), sharedInputs=True)
    ),
    geometry=aoi,
    scale=30,
    maxPixels=1e8,
    bestEffort=True,
)

stats_dict = stats.getInfo()

summary_rows = []
for name in ["NDVI", "NDWI", "SAVI", "NBR"]:
    summary_rows.append({
        "Index": name,
        "Mean": stats_dict.get(f"{name}_mean"),
        "StdDev": stats_dict.get(f"{name}_stdDev"),
        "Min": stats_dict.get(f"{name}_min"),
        "Max": stats_dict.get(f"{name}_max"),
    })

summary_df = pd.DataFrame(summary_rows)
summary_df

In [ ]:
#range check
sample_fc = indices.sample(
    region=aoi,
    scale=30,
    numPixels=2500,
    seed=42,
    geometries=False,
    tileScale=2,
)

sample_info = sample_fc.getInfo()

sample_df = pd.DataFrame(
    [feature["properties"] for feature in sample_info["features"]]
).dropna()

print("Number of sampled clear pixels:", len(sample_df))
sample_df.head()

In [ ]:
# Plot a separate histogram for each index.
for name in ["NDVI", "NDWI", "SAVI", "NBR"]:
    plt.figure(figsize=(7, 4))
    plt.hist(sample_df[name], bins=40)
    plt.xlabel(name)
    plt.ylabel("Number of sampled pixels")
    plt.title(f"Distribution of {name}")
    plt.grid(alpha=0.2)
    plt.show()

In [ ]:
sample_df["NDVI_minus_SAVI"] = sample_df["NDVI"] - sample_df["SAVI"]

comparison = sample_df[["NDVI", "SAVI", "NDVI_minus_SAVI"]].describe()
comparison

In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(sample_df["NDVI"], sample_df["SAVI"], s=8, alpha=0.35)

plt.xlabel("NDVI")
plt.ylabel("SAVI")
plt.title("NDVI versus SAVI")
plt.axline((0, 0), slope=1, linestyle="--")
plt.grid(alpha=0.2)
plt.show()

In [ ]:
vegetation_candidate = ndvi.gt(0.40).rename("Vegetation Candidate")
water_candidate = ndwi.gt(0.00).rename("Water_Candidate")

MaskMap = geemap.Map()
MaskMap.centerObject(aoi, 10)
MaskMap.addLayer(image.clip(aoi), true_color, "True Color", True)
# MaskMap.addLayer(
#     vegetation_candidate.selfMask().clip(aoi),
#     {"palette":['00AA00']},
#     "NDVI > 0.40",
#     False,
# )
MaskMap.addLayer(
    water_candidate.selfMask().clip(aoi),
    {"palette":['00AA00']},
    "NDWI > 0",
    False,
)
MaskMap

In [ ]:
export_indices = index_image.select(
    ['NDVI', 'NDWI', 'SAVI', 'NBR']
).clip(aoi)

task = ee.batch.Export.image.toDrive(
    image=export_indices,
    description="Landsat8_Spectral_Indices",
    folder="GEE_exports",
    fileNamePrefix="Landsat8_NDVI_NDWI_SAVI_NBR",
    region=aoi,
    scale=30,
    maxPixels=1e9,
    fileFormat="GeoTIFF"
)

task.start()

print("Export prepared. Uncomment task.start() to submit it")